In [4]:
"""
Random Forest Supervised Baseline — BETH Dataset
  Purpose : Train a classical ML model (Random Forest) on the same data
and produce identical output plots as the LSTM-DQN so you can
directly compare and argue WHY RL is superior for modern cyber-threat detection.

"""

# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pickle, random, warnings
warnings.filterwarnings('ignore')
import os
import glob

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support, log_loss)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Hyper-parameters ──────────────────────────────────────────────────────────
MAX_ROWS    = 50_000
SEQ_LEN     = 10          # kept so features stay identical to LSTM-DQN
N_ESTIMATORS = 200        # total trees — we grow them incrementally for curves
N_JOBS      = -1          # use all CPU cores
MODEL_SAVE  = 'rf_model.pkl'

POSSIBLE = [
    "E:\MINI Project\IDS2\BETH dataset\labelled_2021may-ip-10-100-1-4.csv"
]
DATA_PATH = next((p for p in POSSIBLE if os.path.exists(p)), None)
if DATA_PATH is None:
    found = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    DATA_PATH = found[0] if found else None
assert DATA_PATH, 'No CSV found. Check dataset path.'
MAX_ROWS = 5_000
SEQ_LEN = 10
MODEL_SAVE = 'lstm_dqn_keras_beth.h5'

# Load data
df_raw = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df_raw.shape}')
df_raw.head(3)

Full dataset shape: (485241, 13)


,timestamp,processId,parentProcessId,userId,processName,hostName,eventId,eventName,argsNum,returnValue,args,sus,evil
0,131.874057,382,1,101,systemd-resolve,ip-10-100-1-4,1005,security_file_open,4,0,"[{'name': 'pathname', 'type': 'const char*', '...",0,0
1,131.874597,382,1,101,systemd-resolve,ip-10-100-1-4,257,openat,4,15,"[{'name': 'dirfd', 'type': 'int', 'value': -10...",0,0
2,131.874796,382,1,101,systemd-resolve,ip-10-100-1-4,5,fstat,2,0,"[{'name': 'fd', 'type': 'int', 'value': 15}, {...",0,0


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# 2.  ATTACK-TYPE LABELLING  (identical to LSTM-DQN script)
# ─────────────────────────────────────────────────────────────────────────────
PROC_INJ_EVENTS  = {'clone', 'kill', 'memfd_create', 'mknod'}
C2_BEACON_EVENTS = {'connect', 'socket', 'bind', 'getsockname',
                    'accept', 'accept4', 'listen', 'dup', 'dup2', 'dup3'}
 
def assign_attack_type(row):
    evt, sus, evil = row['eventName'], row['sus'], row['evil']
    if evil == 1:
        return 'Process_Injection' if evt in PROC_INJ_EVENTS else 'Privilege_Escalation'
    if sus == 1:
        return 'C2_Beaconing' if evt in C2_BEACON_EVENTS else 'Data_Exfiltration'
    return 'Normal'
 
df_raw['attack_type'] = df_raw.apply(assign_attack_type, axis=1)
 
df_s = df_raw.sort_values(['processId', 'timestamp']).copy()
df_s['prev_attack'] = df_s.groupby('processId')['attack_type'].shift(1)
 
def assign_subtype(row):
    if row['attack_type'] == 'Normal':
        return 'normal'
    return 'sequential' if row['prev_attack'] == row['attack_type'] else 'direct'
 
df_s['attack_subtype'] = df_s.apply(assign_subtype, axis=1)
df_s.drop(columns=['prev_attack'], inplace=True)
 
print('\nAttack type distribution:')
print(df_s['attack_type'].value_counts())
print('\nAttack subtype distribution:')
print(df_s['attack_subtype'].value_counts())
 
# ─────────────────────────────────────────────────────────────────────────────
# 3.  STRATIFIED SAMPLING  (identical to LSTM-DQN)
# ─────────────────────────────────────────────────────────────────────────────
df_s['strata'] = df_s['attack_type'] + '_' + df_s['attack_subtype']
target_per_stratum = MAX_ROWS // df_s['strata'].nunique()
 
sampled = (
    df_s.groupby('strata', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), target_per_stratum),
                                  random_state=SEED))
)
if len(sampled) < MAX_ROWS:
    remaining = df_s[~df_s.index.isin(sampled.index)]
    extra = remaining.sample(min(MAX_ROWS - len(sampled), len(remaining)),
                             random_state=SEED)
    sampled = pd.concat([sampled, extra])
 
sampled = sampled.reset_index(drop=True)
print(f'\nFinal sample: {sampled.shape}')
print(sampled['attack_type'].value_counts())
 
# ─────────────────────────────────────────────────────────────────────────────
# 4.  FEATURE ENGINEERING — FLAT, NO SEQUENCES
#     Exactly the same columns as LSTM-DQN, but each row is one sample.
#     Random Forest sees NO temporal context whatsoever.
# ─────────────────────────────────────────────────────────────────────────────
NUMERIC_COLS = ['timestamp', 'processId', 'parentProcessId', 'userId',
                'eventId', 'argsNum', 'returnValue']
 
df_feat = sampled.copy()
 
le_event   = LabelEncoder()
le_process = LabelEncoder()
df_feat['eventName_enc']   = le_event.fit_transform(df_feat['eventName'])
df_feat['processName_enc'] = le_process.fit_transform(df_feat['processName'])
 
subtype_map = {'normal': 0, 'direct': 1, 'sequential': 2}
df_feat['subtype_enc'] = df_feat['attack_subtype'].map(subtype_map)
 
le_label    = LabelEncoder()
df_feat['label'] = le_label.fit_transform(df_feat['attack_type'])
NUM_CLASSES = len(le_label.classes_)
print('\nClasses:', le_label.classes_)
 
FEATURE_COLS = NUMERIC_COLS + ['eventName_enc', 'processName_enc',
                               'sus', 'evil', 'subtype_enc']
FEAT_DIM     = len(FEATURE_COLS)
print(f'Feature dim: {FEAT_DIM}  (flat, no time window)')
 
# ── Scale ─────────────────────────────────────────────────────────────────────
X = df_feat[FEATURE_COLS].values.astype(np.float32)
y = df_feat['label'].values
 
scaler = StandardScaler()
X      = scaler.fit_transform(X)
 
# ── Train / test split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f'\nTrain: {X_train.shape}  |  Test: {X_test.shape}')
 
# ─────────────────────────────────────────────────────────────────────────────
# 5.  INCREMENTAL TRAINING — grow one tree at a time for learning curves
#     (mirrors the DQN's episode-by-episode loss/accuracy tracking)
# ─────────────────────────────────────────────────────────────────────────────
print(f'\nTraining Random Forest ({N_ESTIMATORS} trees, flat features) …')
 
CHECKPOINTS   = list(range(10, N_ESTIMATORS + 1, 10))
train_accs, test_accs     = [], []
train_losses, test_losses = [], []
oob_errors                = []
step_losses               = []   # per-tree test log-loss → mimics step-level loss
 
rf = RandomForestClassifier(
    n_estimators    = 1,
    warm_start      = True,      # add one tree at a time
    oob_score       = True,
    random_state    = SEED,
    n_jobs          = N_JOBS,
    class_weight    = 'balanced',
    max_depth       = 20,
    min_samples_leaf = 2,
)
 
for n_trees in range(1, N_ESTIMATORS + 1):
    rf.set_params(n_estimators=n_trees)
    rf.fit(X_train, y_train)
 
    prob_test    = rf.predict_proba(X_test)
    tree_logloss = log_loss(y_test, prob_test)
    step_losses.append(tree_logloss)
 
    if n_trees in CHECKPOINTS:
        prob_train = rf.predict_proba(X_train)
        tr_loss    = log_loss(y_train, prob_train)
        tr_acc     = accuracy_score(y_train, np.argmax(prob_train, axis=1))
        te_acc     = accuracy_score(y_test,  np.argmax(prob_test,  axis=1))
 
        train_losses.append(tr_loss);   test_losses.append(tree_logloss)
        train_accs.append(tr_acc);      test_accs.append(te_acc)
        oob_errors.append(1 - rf.oob_score_)
 
        print(f'  Trees {n_trees:3d} | Train Acc {tr_acc:.4f} | '
              f'Test Acc {te_acc:.4f} | LogLoss {tree_logloss:.4f} | '
              f'OOB Err {1 - rf.oob_score_:.4f}')
 
print('\nTraining complete!')
 
# Final test predictions
y_prob = rf.predict_proba(X_test)
preds  = np.argmax(y_prob, axis=1)
trues  = y_test
 
# ─────────────────────────────────────────────────────────────────────────────
# 6.  PLOT 1 — TRAINING CURVES
#     Same 2×2 layout as lstm_dqn_keras_training_curves.png
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Random Forest (Flat Supervised) — Training Curves',
             fontsize=14, fontweight='bold')
 
# (0,0) Accuracy over trees
axes[0, 0].plot(CHECKPOINTS, train_accs, color='steelblue', linewidth=2,
                marker='o', markersize=4, label='Train')
axes[0, 0].plot(CHECKPOINTS, test_accs,  color='orange',    linewidth=2,
                marker='s', markersize=4, linestyle='--', label='Test')
axes[0, 0].set_title('Accuracy vs Number of Trees', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Trees')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
 
# (0,1) OOB error
axes[0, 1].plot(CHECKPOINTS, oob_errors, color='seagreen', linewidth=2,
                marker='D', markersize=4)
axes[0, 1].set_title('Out-of-Bag Error vs Number of Trees',
                      fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Trees')
axes[0, 1].set_ylabel('OOB Error Rate')
axes[0, 1].grid(True, alpha=0.3)
 
# (1,0) Per-tree log-loss  →  mirrors "Loss Function Curve (per training step)"
axes[1, 0].plot(range(1, N_ESTIMATORS + 1), step_losses,
                color='tomato', alpha=0.7, linewidth=1)
axes[1, 0].set_title('Log-Loss per Tree Added (per training step)',
                      fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Trees')
axes[1, 0].set_ylabel('Log-Loss')
axes[1, 0].grid(True, alpha=0.3)
 
# (1,1) Smoothed log-loss  →  mirrors "Smoothed Loss Function Curve"
wsize    = max(10, N_ESTIMATORS // 20)
smoothed = pd.Series(step_losses).rolling(window=wsize).mean()
axes[1, 1].plot(range(1, N_ESTIMATORS + 1), step_losses,
                color='tomato', alpha=0.3, linewidth=0.8, label='Raw Log-Loss')
axes[1, 1].plot(range(1, N_ESTIMATORS + 1), smoothed,
                color='darkred', linewidth=2, label=f'Smoothed (w={wsize})')
axes[1, 1].set_title('Smoothed Log-Loss Curve', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Trees')
axes[1, 1].set_ylabel('Log-Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig('rf_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → rf_training_curves.png')
 
# ─────────────────────────────────────────────────────────────────────────────
# 7.  CLASSIFICATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
accuracy              = accuracy_score(trues, preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    trues, preds, average='weighted')
 
print("\n" + "="*60)
print("Classification Report — Random Forest (Flat)")
print("="*60)
print(classification_report(trues, preds, target_names=le_label.classes_))
 
print("Overall Performance Metrics")
print("="*60)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
 
prec_pc, rec_pc, f1_pc, sup_pc = precision_recall_fscore_support(trues, preds)
print("\nPer-Class Metrics")
print("="*60)
for i, cn in enumerate(le_label.classes_):
    print(f"{cn:22s}: Precision={prec_pc[i]:.3f}  Recall={rec_pc[i]:.3f}  "
          f"F1={f1_pc[i]:.3f}  Support={sup_pc[i]}")
 
# ─────────────────────────────────────────────────────────────────────────────
# 8.  PLOT 2 — CONFUSION MATRIX
#     Identical 1×3 layout to lstm_dqn_keras_confusion_matrix.png
# ─────────────────────────────────────────────────────────────────────────────
cm            = confusion_matrix(trues, preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_percent    = cm_normalized * 100
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Random Forest (Flat Supervised) — Confusion Matrix',
             fontsize=14, fontweight='bold')
 
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=le_label.classes_, yticklabels=le_label.classes_,
            cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Counts)',              fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted');  axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30)
 
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=le_label.classes_, yticklabels=le_label.classes_,
            cbar_kws={'label': 'Proportion'})
axes[1].set_title('Normalized Confusion Matrix (Row-wise)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted');  axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=30)
 
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Greens', ax=axes[2],
            xticklabels=le_label.classes_, yticklabels=le_label.classes_,
            cbar_kws={'label': 'Percentage (%)'})
axes[2].set_title('Confusion Matrix (Percentage)',          fontsize=12, fontweight='bold')
axes[2].set_xlabel('Predicted');  axes[2].set_ylabel('True')
axes[2].tick_params(axis='x', rotation=30)
 
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → rf_confusion_matrix.png')
 
# ─────────────────────────────────────────────────────────────────────────────
# 9.  PLOT 3 — LOSS ANALYSIS
#     Identical 2×2 layout to lstm_dqn_keras_loss_analysis.png
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Random Forest (Flat Supervised) — Loss Analysis',
             fontsize=14, fontweight='bold')
 
# (0,0) Loss evolution with moving average
ma50 = pd.Series(step_losses).rolling(window=50).mean()
axes[0, 0].plot(step_losses, color='lightcoral', alpha=0.6, label='Raw Log-Loss')
axes[0, 0].plot(ma50, color='darkred', linewidth=2, label='MA (50 trees)')
axes[0, 0].set_title('Log-Loss Evolution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Trees')
axes[0, 0].set_ylabel('Log-Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
 
# (0,1) Loss distribution histogram
axes[0, 1].hist(step_losses, bins=40, color='steelblue',
                edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Log-Loss Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Log-Loss Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)
 
# (1,0) Train vs Test loss per checkpoint  →  "Average Loss per Episode"
axes[1, 0].plot(CHECKPOINTS, train_losses, color='darkorange', linewidth=2,
                marker='o', markersize=4, label='Train')
axes[1, 0].plot(CHECKPOINTS, test_losses,  color='royalblue',  linewidth=2,
                marker='s', markersize=4, linestyle='--', label='Test')
axes[1, 0].set_title('Average Log-Loss per Checkpoint',
                      fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Trees')
axes[1, 0].set_ylabel('Log-Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
 
# (1,1) Final 200 trees log-loss  →  "Final 200 Steps Loss"
final_losses = step_losses[-200:] if len(step_losses) >= 200 else step_losses
start_idx    = N_ESTIMATORS - len(final_losses) + 1
x_axis       = range(start_idx, N_ESTIMATORS + 1)
axes[1, 1].plot(list(x_axis), final_losses, color='purple', linewidth=1, alpha=0.8)
axes[1, 1].axhline(y=np.mean(final_losses), color='red', linestyle='--',
                   label=f'Mean: {np.mean(final_losses):.4f}')
axes[1, 1].set_title('Final 200 Trees Log-Loss', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Tree Index')
axes[1, 1].set_ylabel('Log-Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig('rf_loss_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → rf_loss_analysis.png')
 
# ─────────────────────────────────────────────────────────────────────────────
# 10. PLOT 4 — FEATURE IMPORTANCES
# ─────────────────────────────────────────────────────────────────────────────
importances = rf.feature_importances_
top_n   = min(20, FEAT_DIM)
top_idx = np.argsort(importances)[::-1][:top_n]
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), importances[top_idx[::-1]], color='steelblue', alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels([FEATURE_COLS[i] for i in top_idx[::-1]], fontsize=9)
ax.set_title(f'Top {top_n} Feature Importances — Random Forest (Flat)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Gini Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → rf_feature_importance.png')
 
# ─────────────────────────────────────────────────────────────────────────────
# 11. SAVE
# ─────────────────────────────────────────────────────────────────────────────
with open(MODEL_SAVE, 'wb') as f:
    pickle.dump(rf, f)
print(f'\nModel saved → {MODEL_SAVE}')
 
preprocessing = {
    'scaler': scaler, 'le_label': le_label,
    'le_event': le_event, 'le_process': le_process,
    'feature_cols': FEATURE_COLS, 'num_classes': NUM_CLASSES,
    'feat_dim': FEAT_DIM
}
with open('preprocessing_rf.pkl', 'wb') as f:
    pickle.dump(preprocessing, f)
print('Preprocessing saved → preprocessing_rf.pkl')
 
# ─────────────────────────────────────────────────────────────────────────────
# 12. FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Training & Evaluation Complete — Random Forest (Flat)")
print("="*60)
print(f"Test Accuracy             : {accuracy:.4f}")
print(f"Final OOB Error           : {1 - rf.oob_score_:.4f}")
print(f"Avg Log-Loss (last 50)    : {np.mean(step_losses[-50:]):.4f}")
print(f"Total Trees               : {N_ESTIMATORS}")
print(f"Feature Dim (flat, no seq): {FEAT_DIM}")